# 10 — Mini MotionSense-AI Project

**Goal:** To collectively use the skills learnt so far together in one small workflow.

## Mini Workflow

This notebook follows a simple data science path:

1. Load data
2. Inspect data
3. Select important columns
4. Filter rows
5. Clean missing values
6. Create magnitude features
7. Summarize activity patterns

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path('../data/raw/motionsense_raw.csv')

# Load the raw MotionSense-style sensor dataset.
df = pd.read_csv(DATA_PATH)
df.head()

,row_id,subject,time_step,activity,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,device
0,1,3,0,Walking,-0.190664,0.687583,1.129328,0.005386,-0.143240,0.014062,phone_A
1,2,24,1,Walking,0.010593,0.542629,0.998704,0.316734,0.085557,0.007263,phone_B
2,3,13,2,Walking,0.221782,0.375365,1.277480,0.114523,0.096630,-0.005492,phone_A
3,4,25,3,Walking,0.167558,0.386942,1.495453,0.203002,-0.047116,-0.038735,phone_A
4,5,11,4,Walking,0.330682,0.558338,1.472442,0.455581,-0.044706,-0.056347,phone_A


In [3]:
# 1. Inspect the dataset size and target labels.
print('Rows and columns:', df.shape)
print('Activity counts:')
print(df['activity'].value_counts())

Rows and columns: (1320, 11)
Activity counts:
activity
Walking               220
Walking Upstairs      220
Walking Downstairs    220
Sitting               220
Standing              220
Laying                220
Name: count, dtype: int64


In [4]:
# 2. Select useful columns.
selected_columns = [
    'row_id', 'subject', 'time_step', 'activity',
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
]

project_df = df[selected_columns].copy()
project_df.head()

,row_id,subject,time_step,activity,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z
0,1,3,0,Walking,-0.190664,0.687583,1.129328,0.005386,-0.143240,0.014062
1,2,24,1,Walking,0.010593,0.542629,0.998704,0.316734,0.085557,0.007263
2,3,13,2,Walking,0.221782,0.375365,1.277480,0.114523,0.096630,-0.005492
3,4,25,3,Walking,0.167558,0.386942,1.495453,0.203002,-0.047116,-0.038735
4,5,11,4,Walking,0.330682,0.558338,1.472442,0.455581,-0.044706,-0.056347


In [5]:
# 3. Clean missing sensor values using the mean.
sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']

for col in sensor_cols:
    project_df[col] = project_df[col].fillna(project_df[col].mean())

print(project_df[sensor_cols].isnull().sum())

acc_x     0
acc_y     0
acc_z     0
gyro_x    0
gyro_y    0
gyro_z    0
dtype: int64


In [6]:
# 4. Create useful magnitude features.
project_df['acc_magnitude'] = np.sqrt(
    project_df['acc_x']**2 + project_df['acc_y']**2 + project_df['acc_z']**2
)

project_df['gyro_magnitude'] = np.sqrt(
    project_df['gyro_x']**2 + project_df['gyro_y']**2 + project_df['gyro_z']**2
)

project_df.head()

,row_id,subject,time_step,activity,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,acc_magnitude,gyro_magnitude
0,1,3,0,Walking,-0.190664,0.687583,1.129328,0.005386,-0.143240,0.014062,1.335853,0.144029
1,2,24,1,Walking,0.010593,0.542629,0.998704,0.316734,0.085557,0.007263,1.136647,0.328166
2,3,13,2,Walking,0.221782,0.375365,1.277480,0.114523,0.096630,-0.005492,1.349830,0.149943
3,4,25,3,Walking,0.167558,0.386942,1.495453,0.203002,-0.047116,-0.038735,1.553763,0.211967
4,5,11,4,Walking,0.330682,0.558338,1.472442,0.455581,-0.044706,-0.056347,1.609092,0.461224


In [7]:
# 5. Summarize movement by activity.
activity_summary = (
    project_df
    .groupby('activity')
    .agg(
        observations=('activity', 'count'),
        average_acc_magnitude=('acc_magnitude', 'mean'),
        average_gyro_magnitude=('gyro_magnitude', 'mean')
    )
    .round(3)
    .sort_values(by='average_acc_magnitude', ascending=False)
)

activity_summary

,observations,average_acc_magnitude,average_gyro_magnitude
activity,,,
Walking Upstairs,220,1.397,0.325
Walking Downstairs,220,1.308,0.295
Walking,220,1.184,0.280
Standing,220,1.003,0.040
Sitting,220,0.997,0.045
Laying,220,0.253,0.031


In [8]:
# 6. Save the cleaned dataset.
# Later we can use processed data from the data/processed folder.
OUTPUT_PATH = Path('../data/processed/motionsense_feature_engineering_output.csv')
project_df.to_csv(OUTPUT_PATH, index=False)

print('Saved file:', OUTPUT_PATH)

Saved file: ..\data\processed\motionsense_feature_engineering_output.csv


## Final Takeaway

Built the foundation:

- NumPy helps calculate with sensor numbers.
- Pandas helps organize and inspect tables.
- Feature engineering turns raw sensor readings into useful project features.